# ЛР1 · 05. Сводка: сравнение, выбор лучшей модели, анализ ошибок, новый срез

Что делает ноутбук (протокол — `PROTOCOL.md`, п. 6–7):
1. собирает журналы всех прогонов (`runs.csv` из lab-00 … lab-03b) в одну таблицу;
2. сравнивает модели: метрики с 95% ДИ, чувствительность к контексту, промпту, режиму;
3. применяет **заранее заданное правило выбора** и сохраняет выбранную модель в `selection.json`;
4. парный бутстрэп: насколько остальные хуже выбранной и значимо ли это;
5. анализ ошибок выбранной модели: кластер, размер станции, час, будни/выходные, особые дни, погода, 10 характерных ошибок;
6. **новый срез**: выбранная модель на 384 станциях, которые не участвовали в выборе, и на двух праздниках
   (4 июля и День труда), которых нет среди точек прогноза теста.

**Kaggle:** Inputs — датасет `vkr-transit-lab` и **вывод всех ноутбуков**: `lab-00-setup`, `lab-01-ts-zeroshot`,
`lab-02-llm-qwen`, `lab-02-llm-phi`, `lab-03-ts-finetune`, `lab-03b-lightgbm`.
Accelerator — **GPU T4** (если выбрана модель без дообучения, хватит и CPU); Internet — **On**.

## 0. Установка и данные

In [ ]:
import subprocess, sys
def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    print("pip install", *args, "->", "OK" if r.returncode == 0 else "ОШИБКА\n" + r.stderr[-1500:])
pip("chronos-forecasting", "peft")
# В образе Kaggle стоит torchao 0.10, а peft (LoRA) требует либо torchao ≥ 0.16, либо его отсутствия:
# первый запуск LoRA упал с ImportError. Нам torchao не нужен — удаляем.
r = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], capture_output=True, text=True)
print("pip uninstall torchao ->", "OK" if r.returncode == 0 else r.stderr[-500:])

In [ ]:
import glob, json, os, re, sys, time, gc
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

hits = glob.glob("/kaggle/input/**/labkit.py", recursive=True) or glob.glob("labkit.py")
LAB00 = Path(hits[0]).parent
sys.path.insert(0, str(LAB00))
import labkit as lk

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("labkit:", LAB00, "| device:", DEVICE)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
OUT = lk.results_dir()
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})
C_BEST, C_2, C_3, C_4, C_NAIVE = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8c8c8c"
from matplotlib.ticker import NullFormatter, FuncFormatter

def log_axis(ax, axis="y"):
    """Логарифмическая шкала с понятными подписями 0.02, 0.05, 0.1 … вместо степеней десяти."""
    a = ax.yaxis if axis == "y" else ax.xaxis
    lim = ax.get_ylim() if axis == "y" else ax.get_xlim()
    a.set_minor_formatter(NullFormatter())
    a.set_ticks([t for t in [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1, 2] if lim[0] <= t <= lim[1]])
    a.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    (ax.set_ylim if axis == "y" else ax.set_xlim)(lim)

d = lk.load()
sub = json.load(open(LAB00 / "subset.json", encoding="utf-8"))
stations = sub["stations"]
test_orig = [pd.Timestamp(x) for x in sub["test_origins"]]
assert stations == lk.make_subset(d), "подвыборка не совпадает с lab-00!"
tmax = d.train_max()
st = d.static
print(f"станций в данных: {d.y.shape[1]}, в подвыборке: {len(stations)}, точек прогноза: {len(test_orig)}")

## 1. Сбор журналов всех прогонов

Каждый ноутбук писал свой `results/runs.csv` и файлы прогнозов `<модель>__<режим>.parquet`.
Если одна конфигурация встречается дважды, берётся последний по дате прогон.

In [ ]:
run_files = [Path(p) for p in glob.glob("/kaggle/input/**/runs.csv", recursive=True)]
if not run_files:                                    # локальный запуск из папки notebooks/ репозитория
    run_files = [Path(p) for p in glob.glob("../results/**/runs.csv", recursive=True)]
run_files = [p for p in run_files if OUT.resolve() not in p.resolve().parents]

parts = []
for p in run_files:
    r = pd.read_csv(p)
    r["src"] = str(p.parent)
    parts.append(r)
    print(f"{p.parent}: {len(r)} прогонов, модели: {sorted(r['model'].unique())}")
runs = pd.concat(parts, ignore_index=True)
runs = runs.sort_values("date").drop_duplicates(["model", "mode"], keep="last").reset_index(drop=True)

NAMES = {"chronos_bolt": "Chronos-Bolt Base", "chronos2": "Chronos-2", "timesfm25": "TimesFM 2.5",
         "moirai2_small": "Moirai 2.0 Small", "lag_llama": "Lag-Llama", "ttm_r2": "TTM r2",
         "qwen25_3b": "Qwen2.5-3B", "phi35_mini": "Phi-3.5-mini",
         "seasonal_naive": "Сезонный наивный", "lightgbm": "LightGBM"}
FAMILY = {"qwen25_3b": "LLM", "phi35_mini": "LLM", "seasonal_naive": "baseline", "lightgbm": "baseline"}

def regime(mode):
    for pre, name in [("peft_lora", "PEFT (LoRA)"), ("fs5pct", "few-shot: дообучение на 5% train"),
                      ("fs3", "few-shot: 3 примера в промпте"), ("cov", "ковариаты"), ("graph", "соседи по графу"),
                      ("zs", "zero-shot"), ("baseline", "ориентир"), ("trained", "обучение с нуля")]:
        if mode.startswith(pre):
            return name
    return mode

runs["family"] = runs["model"].map(FAMILY).fillna("TS")
runs["name"] = runs["model"].map(NAMES).fillna(runs["model"])
runs["regime"] = runs["mode"].map(regime)
runs["ctx"] = runs["mode"].str.extract(r"ctx(\d+)")[0].astype(float)
runs["prompt"] = runs["mode"].str.extract(r"_(P\d)_")[0]
runs["part"] = np.where(runs["mode"].str.contains("newslice"), "newslice",
                        np.where(runs["mode"].str.endswith("_val"), "val", "test"))
runs.to_csv(OUT / "all_runs.csv", index=False)

def files_of(model, mode):
    src = Path(runs.loc[(runs.model == model) & (runs["mode"] == mode), "src"].iat[0])
    return src / f"{model}__{mode}.parquet", src / f"{model}__{mode}__windows.parquet"

T = runs[runs["part"] == "test"].copy()
print(f"\nВсего конфигураций на тесте: {len(T)}; моделей: {T['model'].nunique()}")
print(T.groupby("name")["mode"].apply(list).to_string())

## 2. WAPE в часы пик

Для правила выбора нужен WAPE в часы пик (7–9 и 16–19 ч, будни). Он считается по сохранённым почасовым прогнозам.
Поскольку все точки прогноза — 00:00, шаг горизонта h соответствует часу суток h − 1.

In [ ]:
PEAK_HOURS = [7, 8, 9, 16, 17, 18, 19]

def load_long(model, mode):
    f, _ = files_of(model, mode)
    L = pd.read_parquet(f)
    L["origin"] = pd.to_datetime(L["origin"])
    L["hour"] = L["step"] - 1
    L["ts"] = L["origin"] + pd.to_timedelta(L["hour"], unit="h")
    return L

def wape(L, col="yhat"):
    m = L["in_metric"].astype(bool)
    return float((L.loc[m, "y"] - L.loc[m, col]).abs().sum() / L.loc[m, "y"].sum())

peak = []
for _, r in T.iterrows():
    L = load_long(r["model"], r["mode"])
    pk = L[L["hour"].isin(PEAK_HOURS) & (L["origin"].dt.dayofweek < 5)]
    peak.append(wape(pk))
T["wape_peak"] = peak

## 3. Сводная таблица (тест, 40 станций × 15 точек)

In [ ]:
SHOW = ["name", "mode", "n_windows", "invalid_share", "invalid_strict_share", "wape", "wape_ci_lo", "wape_ci_hi",
        "wape_peak", "mae", "rmse", "mase", "wql", "cov80", "ms_p50", "ms_p95", "gpu_peak_gb"]
table = T.sort_values("wape")[[c for c in SHOW if c in T]]
table.to_csv(OUT / "table_test_all.csv", index=False)
print(table.round(4).to_string(index=False))

NAIVE = float(T.loc[T["model"] == "seasonal_naive", "wape"].iat[0])
print(f"\nСезонный наивный прогноз: WAPE = {NAIVE:.4f}")

### 3.1. Главная таблица: одна строка на модель

Для каждой модели — основной zero-shot (контекст 336 ч, у LLM промпт P2) и её лучшая конфигурация на тесте.

In [ ]:
main_mode = {m: ("zs_P2_ctx336_test" if f == "LLM" else "baseline_test" if m == "seasonal_naive"
                 else "trained_test" if m == "lightgbm" else "zs_ctx336_test")
             for m, f in T.groupby("model")["family"].first().items()}
rows = []
for m, g in T.groupby("model"):
    base = g[g["mode"] == main_mode[m]]
    best = g.sort_values("wape").iloc[0]
    b = base.iloc[0] if len(base) else best
    rows.append({"Модель": NAMES.get(m, m), "Тип": g["family"].iat[0],
                 "WAPE zero-shot [95% ДИ]": f"{b.wape:.3f} [{b.wape_ci_lo:.3f}; {b.wape_ci_hi:.3f}]",
                 "Лучшая конфигурация": best["mode"],
                 "WAPE лучшей [95% ДИ]": f"{best.wape:.3f} [{best.wape_ci_lo:.3f}; {best.wape_ci_hi:.3f}]",
                 "MASE": round(best.mase, 3), "покрытие 80%": round(best.cov80, 3) if pd.notna(best.get("cov80")) else "—",
                 "невалидных": f"{best.invalid_share:.1%}",
                 "p95, мс/окно": round(best.ms_p95, 1) if pd.notna(best.ms_p95) else "—",
                 "GPU, ГБ": round(best.gpu_peak_gb, 2) if pd.notna(best.gpu_peak_gb) else "—",
                 "_wape": best.wape})
main = pd.DataFrame(rows).sort_values("_wape").drop(columns="_wape")
main.to_csv(OUT / "table_main.csv", index=False)
main

In [ ]:
# график: WAPE лучшей конфигурации каждой модели с 95% ДИ
g = T.sort_values("wape").groupby("model").head(1).sort_values("wape", ascending=False)
fig, ax = plt.subplots(figsize=(8, 4.2))
y = np.arange(len(g))
colors = [C_NAIVE if f == "baseline" else C_2 if f == "LLM" else C_BEST for f in g["family"]]
ax.errorbar(g["wape"], y, xerr=[g["wape"] - g["wape_ci_lo"], g["wape_ci_hi"] - g["wape"]], fmt="none",
            ecolor="#555555", elinewidth=1, capsize=3)
ax.scatter(g["wape"], y, c=colors, s=40, zorder=3)
ax.axvline(NAIVE, color=C_NAIVE, ls="--", lw=1)
ax.text(NAIVE, -0.9, " сезонный наивный", color="#555555", fontsize=8)
ax.set_yticks(y, [f"{NAMES.get(m, m)}  ({md_})" for m, md_ in zip(g["model"], g["mode"])])
ax.set_xscale("log"); ax.set_xlabel("WAPE (лог. шкала), меньше — лучше")
log_axis(ax, "x")
ax.set_ylim(-1.2, len(g) - 0.5)
ax.set_title("Лучшая конфигурация каждой модели, тест, 95% ДИ")
plt.tight_layout(); plt.savefig(FIG / "01_wape_ci.png"); plt.show()

## 4. Чувствительность: длина контекста, промпт, режим

In [ ]:
ctx = T[T["mode"].str.match(r"^zs_(P2_)?ctx\d+_test$")].pivot_table(index="name", columns="ctx", values="wape")
ctx["разброс (max − min)"] = ctx.max(axis=1) - ctx.min(axis=1)
print("WAPE zero-shot по длине контекста, ч:")
print(ctx.round(4).to_string())
ctx.to_csv(OUT / "sens_context.csv")

pr = T[(T["family"] == "LLM") & T["mode"].str.startswith("zs_") & (T["ctx"] == 336)].pivot_table(index="name", columns="prompt", values="wape")
if len(pr):
    pr["разброс (max − min)"] = pr.max(axis=1) - pr.min(axis=1)
    print("\nWAPE текстовых LLM по формулировке промпта (контекст 336 ч):")
    print(pr.round(4).to_string())
    pr.to_csv(OUT / "sens_prompt.csv")

reg = T[T["ctx"].isin([336, 672]) & (T["family"] != "baseline") & ~T["prompt"].isin(["P1", "P3"])].pivot_table(
    index=["name", "ctx"], columns="regime", values="wape")
print("\nWAPE по режимам (zero-shot / few-shot / PEFT / ковариаты / соседи):")
print(reg.round(4).to_string())
reg.to_csv(OUT / "sens_regime.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
pal = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#7b5ea7", "#a0522d"]
for k, (name, row) in enumerate(ctx.drop(columns="разброс (max − min)").iterrows()):
    ax.plot(row.index, row.values, marker="o", lw=2, ms=6, color=pal[k % len(pal)], label=name)
ax.axhline(NAIVE, color=C_NAIVE, ls="--", lw=1, label="сезонный наивный")
ax.set_xticks(list(lk.CONTEXTS)); ax.set_yscale("log")
ax.set_xlabel("длина контекста, ч"); ax.set_ylabel("WAPE (лог. шкала)")
log_axis(ax)
ax.set_title("Zero-shot: чувствительность к длине контекста")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.savefig(FIG / "02_context.png"); plt.show()

## 5. Правило выбора лучшей модели (PROTOCOL.md, п. 7 — зафиксировано до запуска)

1. **Фильтр:** доля невалидных окон ≤ 1% и WAPE не меньше чем на 5% лучше сезонного наивного.
2. **Балл** (min-max среди прошедших фильтр, 1 = лучшая): 50% WAPE, 15% WAPE в пик, 15% |покрытие 80% − 0,8|
   (у моделей без квантилей — худшее среди прошедших), 10% p95 времени на окно, 10% пиковая память GPU.
3. Если разница баллов двух первых < 0,02 — берётся более лёгкая по памяти.

Кандидаты — все конфигурации 8 моделей на тесте (baseline'ы не участвуют).

In [ ]:
cand = T[T["family"] != "baseline"].copy()
cand["pass_invalid"] = cand["invalid_share"] <= 0.01
cand["pass_naive"] = cand["wape"] <= 0.95 * NAIVE
cand["passed"] = cand["pass_invalid"] & cand["pass_naive"]
P = cand[cand["passed"]].copy()
print(f"Прошли фильтр: {len(P)} из {len(cand)} конфигураций")
print("Не прошли:", ", ".join(f"{NAMES[m]} {md_}" for m, md_ in cand.loc[~cand["passed"], ["model", "mode"]].values))

P["calib"] = (P["cov80"] - 0.8).abs()
P["calib"] = P["calib"].fillna(P["calib"].max())
P["gpu"] = P["gpu_peak_gb"].fillna(0.0)

def score01(x):                       # 1 = лучшая (наименьшее значение), 0 = худшая
    return (x.max() - x) / (x.max() - x.min()) if x.max() > x.min() else pd.Series(1.0, index=x.index)

WEIGHTS = {"wape": 0.50, "wape_peak": 0.15, "calib": 0.15, "ms_p95": 0.10, "gpu": 0.10}
for k in WEIGHTS:
    P[f"s_{k}"] = score01(P[k])
P["score"] = sum(w * P[f"s_{k}"] for k, w in WEIGHTS.items())
P = P.sort_values("score", ascending=False)

top = P.iloc[0]
if len(P) > 1 and top["score"] - P.iloc[1]["score"] < 0.02:
    pair = P.iloc[:2].sort_values("gpu")
    print(f"Разница баллов двух первых {top['score'] - P.iloc[1]['score']:.4f} < 0,02 — берётся более лёгкая по памяти")
    top = pair.iloc[0]
BEST = (top["model"], top["mode"])

sel_cols = ["name", "mode", "wape", "wape_peak", "calib", "ms_p95", "gpu", "score"]
print(P[sel_cols].round(4).to_string(index=False))
print(f"\nВЫБРАНА: {NAMES[BEST[0]]}, конфигурация {BEST[1]}: WAPE = {top['wape']:.4f}, балл = {top['score']:.3f}")
P.to_csv(OUT / "selection_table.csv", index=False)
json.dump({"model": BEST[0], "mode": BEST[1], "name": NAMES[BEST[0]], "wape": float(top["wape"]),
           "score": float(top["score"]), "weights": WEIGHTS, "naive_wape": NAIVE},
          open(OUT / "selection.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

## 6. Парный бутстрэп: разница с выбранной моделью

Разность WAPE (конфигурация − выбранная) на одних и тех же 600 окнах, 1 000 повторов.
Если 95% ДИ разности не содержит 0, различие значимо.

In [ ]:
def windows_of(model, mode):
    ws = pd.read_parquet(files_of(model, mode)[1])
    ws["origin"] = pd.to_datetime(ws["origin"])
    return ws

ws_best = windows_of(*BEST)
others = T[(T["model"] != BEST[0]) | (T["mode"] != BEST[1])].sort_values("wape")
others = pd.concat([others.groupby("model").head(1), others[others["model"] == BEST[0]]]).drop_duplicates(["model", "mode"])
rows = []
for _, r in others.iterrows():
    diff, lo, hi = lk.paired_diff_ci(windows_of(r["model"], r["mode"]), ws_best, "wape")
    rows.append({"модель": r["name"], "режим": r["mode"], "WAPE": r["wape"], "разность с выбранной": diff,
                 "ДИ низ": lo, "ДИ верх": hi, "значимо": "да" if lo > 0 or hi < 0 else "нет"})
pairs = pd.DataFrame(rows).sort_values("WAPE")
pairs.to_csv(OUT / "paired_bootstrap.csv", index=False)
print(pairs.round(4).to_string(index=False))

## 7. Анализ ошибок выбранной модели

Срезы: кластер станции, размер станции (квинтиль `scale_weekday` по всем станциям), час суток (= шаг горизонта),
будни / выходные, особые дни (праздник или городское событие — по `covariates_time`; матч или концерт у станции —
по `events_station_hour`), дождь (≥ 5 мм за сутки) и жара (максимум ≥ 30 °C).
Для сравнения рядом — сезонный наивный, LightGBM и лучшая текстовая LLM.

In [ ]:
def daily(col, how):
    return d.cov[col].groupby(d.cov.index.normalize()).agg(how) if col in d.cov else None

day = pd.DataFrame({"rain_mm": daily("precipitation", "sum"), "tmax": daily("temperature_2m", "max"),
                    "holiday": daily("is_holiday", "max"), "city_event": daily("is_city_event", "max")})
ev_path = d.root / "events_station_hour.parquet"
if ev_path.exists():
    ev = pd.read_parquet(ev_path)
    ev["station"] = ev["station_complex_id"].astype(str)
    ev["date"] = pd.to_datetime(ev["transit_timestamp"]).dt.normalize()
    ev_days = set(map(tuple, ev.loc[ev["ev_today"].fillna(0) > 0, ["station", "date"]].drop_duplicates().values))
else:
    ev_days = set()
scale_q = pd.qcut(st["scale_weekday"], 5, labels=["Q1 (малые)", "Q2", "Q3", "Q4", "Q5 (крупные)"])

def annotate(L):
    L = L.copy()
    L["date"] = L["origin"].dt.normalize()
    L["cluster"] = L["station"].map(st["cluster"]).fillna(-1).astype(int).astype(str)
    L["size"] = L["station"].map(scale_q).astype(str)
    L["day_type"] = np.where(L["origin"].dt.dayofweek >= 5, "выходной", "будний")
    L = L.join(day, on="date")
    L["special"] = np.where(L["holiday"].fillna(0) > 0, "праздник",
                   np.where(L["city_event"].fillna(0) > 0, "городское событие", "обычный"))
    L["station_event"] = [("событие у станции" if (s, dd) in ev_days else "нет") for s, dd in zip(L["station"], L["date"])]
    L["weather"] = np.where(L["rain_mm"] >= 5, "дождь ≥ 5 мм",
                   np.where(L["tmax"] >= 30, "жара ≥ 30 °C", "обычная"))
    return L

KEY = ["station", "origin", "step"]
LB = annotate(load_long(*BEST))
compare = {f"{NAMES[BEST[0]]} (выбранная)": BEST, "Сезонный наивный": ("seasonal_naive", "baseline_test")}
if ((T["model"] == "lightgbm") & (T["mode"] == "trained_test")).any():
    compare["LightGBM"] = ("lightgbm", "trained_test")
llm = T[T["family"] == "LLM"].sort_values("wape")
if len(llm):
    compare[f"{llm['name'].iat[0]} (лучшая LLM)"] = (llm["model"].iat[0], llm["mode"].iat[0])
for label, (m, md_) in compare.items():
    if (m, md_) != BEST:
        LB[label] = load_long(m, md_).set_index(KEY)["yhat"].reindex(pd.MultiIndex.from_frame(LB[KEY])).to_numpy()
LB = LB.rename(columns={"yhat": list(compare)[0]})

def by(col, L=LB, labels=None):
    labels = labels or list(compare)
    rows = {}
    for g, part in L.groupby(col):
        m = part["in_metric"].astype(bool)
        r = {lab: (part.loc[m, "y"] - part.loc[m, lab]).abs().sum() / part.loc[m, "y"].sum() for lab in labels}
        r["окон"] = part.loc[:, ["station", "origin"]].drop_duplicates().shape[0]
        rows[g] = r
    return pd.DataFrame(rows).T

err = {}
for col, title in [("cluster", "кластер"), ("size", "размер станции"), ("day_type", "будни / выходные"),
                   ("special", "особый день"), ("station_event", "событие у станции"), ("weather", "погода")]:
    err[col] = by(col)
    print(f"\nWAPE по срезу «{title}»:")
    print(err[col].round(3).to_string())
pd.concat(err, names=["срез", "группа"]).to_csv(OUT / "errors_by_group.csv")

In [ ]:
hr = by("hour")
fig, ax = plt.subplots(figsize=(7, 3.8))
for lab, c in zip(compare, [C_BEST, C_NAIVE, C_3, C_2]):
    ax.plot(hr.index, hr[lab], marker="o", ms=4, lw=2, color=c, label=lab)
ax.set_yscale("log"); ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("час суток (= шаг горизонта − 1)"); ax.set_ylabel("WAPE (лог. шкала)")
log_axis(ax)
ax.set_title("Ошибка по часам суток")
ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / "03_wape_by_hour.png"); plt.show()
hr.to_csv(OUT / "errors_by_hour.csv")

### 7.1. Десять характерных ошибок

Пять окон с наибольшей абсолютной ошибкой за сутки и пять с наибольшей относительной (среди окон,
где суточный поток выше медианы, — чтобы не брать почти пустые станции).

In [ ]:
W1 = LB.assign(ae=(LB["y"] - LB[list(compare)[0]]).abs().where(LB["in_metric"].astype(bool), 0),
               ae_naive=(LB["y"] - LB["Сезонный наивный"]).abs().where(LB["in_metric"].astype(bool), 0))
agg = W1.groupby(["station", "origin"]).agg(y=("y", "sum"), yhat=(list(compare)[0], "sum"), ae=("ae", "sum"),
                                            ae_naive=("ae_naive", "sum"), open_hours=("in_metric", "sum"),
                                            special=("special", "first"), station_event=("station_event", "first"),
                                            weather=("weather", "first"), cluster=("cluster", "first"),
                                            size=("size", "first")).reset_index()
agg["wape"] = agg["ae"] / agg["y"].clip(lower=1)
agg["wape_naive"] = agg["ae_naive"] / agg["y"].clip(lower=1)
abs_top = agg.nlargest(5, "ae")
rel_top = agg[(agg["y"] > agg["y"].median()) & ~agg.set_index(["station", "origin"]).index.isin(
    abs_top.set_index(["station", "origin"]).index)].nlargest(5, "wape")
top10 = pd.concat([abs_top.assign(вид="абсолютная"), rel_top.assign(вид="относительная")], ignore_index=True)
top10["станция"] = top10["station"].map(st["station_complex"]) if "station_complex" in st else top10["station"]
top10["день"] = top10["origin"].dt.strftime("%d.%m.%Y %a")
cols10 = ["вид", "station", "станция", "день", "cluster", "size", "y", "yhat", "wape", "wape_naive",
          "open_hours", "special", "station_event", "weather"]
top10[cols10].to_csv(OUT / "top10_errors.csv", index=False)
print(top10[cols10].round(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6), sharex=True)
for ax, (_, r) in zip(axes.ravel(), top10.iterrows()):
    part = LB[(LB["station"] == r["station"]) & (LB["origin"] == r["origin"])].sort_values("hour")
    if "q10" in part:
        ax.fill_between(part["hour"], part["q10"], part["q90"], color=C_BEST, alpha=0.15, lw=0, label="80% интервал")
    ax.plot(part["hour"], part["y"], color="#222222", lw=2, label="факт")
    ax.plot(part["hour"], part[list(compare)[0]], color=C_BEST, lw=2, label="выбранная")
    ax.plot(part["hour"], part["Сезонный наивный"], color=C_NAIVE, lw=1, ls="--", label="наивный")
    ax.set_title(f"{str(r['станция'])[:22]}\n{r['день']}, WAPE {r['wape']:.2f}", fontsize=8)
axes[0, 0].legend(fontsize=7)
for ax in axes[1]:
    ax.set_xlabel("час")
plt.suptitle(f"10 характерных ошибок: {NAMES[BEST[0]]} ({BEST[1]})", fontsize=10)
plt.tight_layout(); plt.savefig(FIG / "04_top10_errors.png"); plt.show()

## 8. Новый срез: выбранная модель на станциях, не участвовавших в выборе

- **Новые станции:** 384 станции вне подвыборки, те же 15 точек прогноза (5 760 окон).
- **Праздники:** 4 июля и 7 сентября 2026 (День труда) — их нет среди точек прогноза теста; все 424 станции.

Сначала выбранная конфигурация пересчитывается на исходных 600 окнах и сверяется с сохранённым WAPE —
это проверка воспроизводимости. Для дообученных конфигураций дообучение повторяется с теми же настройками,
что в lab-03 (небольшое расхождение возможно из-за недетерминированности GPU).

In [ ]:
Q = list(lk.QUANTILES)
idx = d.y.index
cols = list(d.y.columns)
Yall = d.y.to_numpy(dtype="float64")
COV_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_holiday", "is_school_day", "is_city_event",
            "temperature_2m", "precipitation", "snowfall"]

def chronos2_base():
    from chronos import Chronos2Pipeline
    return Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)

def chronos2_finetuned(kind):
    """Повтор дообучения из lab-03: последние 59 дней train, все станции, 500 шагов."""
    os.environ["WANDB_DISABLED"] = "true"
    train_end = d.part("train")[1]
    p_end = idx.get_loc(train_end) + 1
    p_ft0 = idx.get_loc(train_end.normalize() - pd.Timedelta(days=58))
    Ymask = np.where(d.closed.to_numpy(), np.nan, Yall)
    series = [Ymask[p_ft0 - 672:p_end, k].astype("float32") for k in range(len(cols))]
    kw = dict(finetune_mode="lora", learning_rate=1e-4) if kind == "lora" else dict(finetune_mode="full", learning_rate=1e-5)
    t0 = time.time()
    ft = chronos2_base().fit(series, prediction_length=lk.HORIZON, context_length=672, num_steps=500, batch_size=64,
                             output_dir=f"/tmp/c2_{kind}", **kw)
    print(f"дообучение Chronos-2 ({kind}) повторено за {(time.time() - t0) / 60:.1f} мин")
    return ft

def c2_predictor(pipe, mode):
    def out(inputs):
        q, _ = pipe.predict_quantiles(inputs, prediction_length=lk.HORIZON, quantile_levels=Q, batch_size=128)
        q = np.stack([x[0].cpu().numpy() for x in q])
        return q[..., Q.index(0.5)], q
    if mode.startswith("cov"):
        COV = d.cov[COV_COLS].to_numpy(dtype="float64")
        def predict(ctx, pos, w, i):
            return out([{"target": ctx[j].astype("float32"),
                         "past_covariates": {n: COV[pos[j] - ctx.shape[1]:pos[j], k] for k, n in enumerate(COV_COLS)},
                         "future_covariates": {n: COV[pos[j]:pos[j] + lk.HORIZON, k] for k, n in enumerate(COV_COLS)}}
                        for j in range(len(ctx))])
    elif mode.startswith("graph"):
        A = np.load(d.root / "graph" / "adj_od_top10.npy")
        def predict(ctx, pos, w, i):
            res = []
            for j in range(len(ctx)):
                k0 = cols.index(w.meta["station"].iat[i + j])
                nb = [k for k in np.argsort(-A[k0]) if A[k0, k] > 0][:10]
                res.append(np.vstack([ctx[j], Yall[pos[j] - ctx.shape[1]:pos[j], nb].T]).astype("float32"))
            return out(res)
    else:
        def predict(ctx, pos, w, i):
            return out(ctx[:, None, :].astype("float32"))
    return predict

def build_predictor(model, mode):
    """Возвращает predict(ctx, pos, w, i) -> (point, quant) для выбранной конфигурации."""
    if model == "chronos2":
        if mode.startswith("peft_lora"):
            pipe = chronos2_finetuned("lora")
        elif mode.startswith("fs5pct"):
            pipe = chronos2_finetuned("full")
        else:
            pipe = chronos2_base()
        return c2_predictor(pipe, mode)
    if model == "chronos_bolt":
        from chronos import BaseChronosPipeline
        bolt = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-base", device_map=DEVICE, torch_dtype=torch.float32)
        def predict(ctx, pos, w, i):
            q, _ = bolt.predict_quantiles(torch.tensor(ctx, dtype=torch.float32), prediction_length=lk.HORIZON,
                                          quantile_levels=Q)
            q = q.cpu().numpy(); return q[..., Q.index(0.5)], q
        return predict
    if model == "moirai2_small":
        pip("--no-deps", "uni2ts==2.0.0"); pip("einops", "jaxtyping", "hydra-core", "lightning", "gluonts")
        from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module
        c = int(re.search(r"ctx(\d+)", mode).group(1))
        mf = Moirai2Forecast(module=Moirai2Module.from_pretrained("Salesforce/moirai-2.0-R-small"),
                             prediction_length=lk.HORIZON, context_length=c, target_dim=1,
                             feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0).to(DEVICE)
        def predict(ctx, pos, w, i):
            o = np.asarray(mf.predict([x.astype("float32") for x in ctx])).reshape(len(ctx), len(Q), lk.HORIZON)
            q = np.transpose(o, (0, 2, 1)); return q[..., Q.index(0.5)], q
        return predict
    if model == "timesfm25":
        pip("timesfm[torch]")
        import timesfm
        tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch", torch_compile=False)
        tfm.compile(timesfm.ForecastConfig(max_context=1024, max_horizon=256, normalize_inputs=True,
                                           use_continuous_quantile_head=True, fix_quantile_crossing=True,
                                           per_core_batch_size=64))
        def predict(ctx, pos, w, i):
            _, q = tfm.forecast(horizon=lk.HORIZON, inputs=[x.astype("float32") for x in ctx])
            q = np.asarray(q)[..., 1:10]; return q[..., Q.index(0.5)], q
        return predict
    raise NotImplementedError(f"Повторный запуск {model} в lab-05 не предусмотрен — возьмите код из lab-01/03")

def forecast(model, mode, w, predict, batch=128):
    """Прогноз по окнам с той же постобработкой, что в lab-01, и запись в runs.csv."""
    pos = np.array([idx.get_loc(o) for o in w.meta["origin"]])
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t = lk.Timer(); pts, qs, sizes = [], [], []
    for i in range(0, len(w), batch):
        with t:
            p, q = predict(w.ctx[i:i + batch], pos[i:i + batch], w, i)
        pts.append(np.asarray(p, dtype="float64")); qs.append(np.asarray(q, dtype="float64"))
        sizes.append(min(batch, len(w) - i))
    point = np.clip(np.concatenate(pts), 0, None)
    quant = np.clip(np.sort(np.concatenate(qs), axis=-1), 0, None)
    valid = lk.check_valid(list(point), w, tmax)
    point = lk.apply_fallback(list(point), valid, w)
    quant[~valid] = point[~valid][..., None]
    per = np.array(t.times) * 1000 / np.array(sizes)
    timing = {"ms_p50": float(np.median(per)), "ms_p95": float(np.quantile(per, 0.95)), "total_s": float(per.sum())}
    return lk.save_run(model, mode, w, point, valid, quant, timing,
                       {"context": w.ctx.shape[1], "n_stations": w.meta["station"].nunique()})

def naive_run(mode, w):
    p = lk.seasonal_naive(w.ctx)
    return lk.save_run("seasonal_naive", mode, w, p, np.ones(len(w), bool), None, {}, {"n_stations": w.meta["station"].nunique()})

In [ ]:
C_SEL = int(re.search(r"ctx(\d+)", BEST[1]).group(1))
predict = build_predictor(*BEST)

# 1) воспроизводимость на исходных окнах
w_sub = lk.make_windows(d, stations, test_orig, context=C_SEL)
rep = forecast(BEST[0], BEST[1].replace("_test", "_repeat_test"), w_sub, predict)
print(f"Повтор на исходных 600 окнах: WAPE {rep['wape']:.4f} (в lab-01/03 было {top['wape']:.4f}, "
      f"разница {rep['wape'] - top['wape']:+.4f})")

# 2) новые станции, те же точки прогноза
new_st = [s for s in cols if s not in set(stations)]
w_new = lk.make_windows(d, new_st, test_orig, context=C_SEL)
w_new_naive = lk.make_windows(d, new_st, test_orig, context=lk.SEASON * 2)
r_new = forecast(BEST[0], f"{BEST[1].replace('_test', '')}_newslice_test", w_new, predict)
n_new = naive_run("baseline_newslice_test", w_new_naive)
print(f"Новый срез ({len(new_st)} станций, {len(w_new)} окон): WAPE {r_new['wape']:.4f} "
      f"[{r_new['wape_ci_lo']:.4f}; {r_new['wape_ci_hi']:.4f}], наивный {n_new['wape']:.4f}")

# 3) праздники, все станции
HOLIDAYS = [pd.Timestamp("2026-07-04"), pd.Timestamp("2026-09-07")]
w_hol = lk.make_windows(d, cols, HOLIDAYS, context=C_SEL)
r_hol = forecast(BEST[0], f"{BEST[1].replace('_test', '')}_holidays_test", w_hol, predict)
n_hol = naive_run("baseline_holidays_test", lk.make_windows(d, cols, HOLIDAYS, context=lk.SEASON * 2))
print(f"Праздники (4.07 и 7.09, {len(w_hol)} окон): WAPE {r_hol['wape']:.4f} "
      f"[{r_hol['wape_ci_lo']:.4f}; {r_hol['wape_ci_hi']:.4f}], наивный {n_hol['wape']:.4f}")

In [ ]:
# сравнение на новом срезе: парный бутстрэп «выбранная − наивный» и «выбранная − LightGBM»
mode_new = f"{BEST[1].replace('_test', '')}_newslice_test"
ws_new = pd.read_parquet(OUT / f"{BEST[0]}__{mode_new}__windows.parquet")
ws_naive_new = pd.read_parquet(OUT / "seasonal_naive__baseline_newslice_test__windows.parquet")
rows = [("Сезонный наивный", *lk.paired_diff_ci(ws_new, ws_naive_new, "wape"))]
lgb_new = runs[(runs["model"] == "lightgbm") & (runs["mode"] == "trained_newslice_test")]
if len(lgb_new):
    ws_lgb_new = windows_of("lightgbm", "trained_newslice_test")
    rows.append(("LightGBM", *lk.paired_diff_ci(ws_new, ws_lgb_new, "wape")))
else:
    print("LightGBM на новом срезе не найден (нужен lab-03b с разделом 4)")
cmp_new = pd.DataFrame(rows, columns=["с чем", "разность WAPE (выбранная − другая)", "ДИ низ", "ДИ верх"])
print(cmp_new.round(4).to_string(index=False))

# подвыборка vs новый срез по кластерам
L_new = annotate(pd.read_parquet(OUT / f"{BEST[0]}__{mode_new}.parquet").assign(
    origin=lambda x: pd.to_datetime(x["origin"]), hour=lambda x: x["step"] - 1))
L_new["Сезонный наивный"] = pd.read_parquet(OUT / "seasonal_naive__baseline_newslice_test.parquet").set_index(KEY)[
    "yhat"].reindex(pd.MultiIndex.from_frame(L_new[KEY])).to_numpy()
L_new = L_new.rename(columns={"yhat": list(compare)[0]})
two = pd.concat({f"подвыборка ({len(stations)})": by("cluster", LB, [list(compare)[0], "Сезонный наивный"]),
                 f"новый срез ({len(new_st)})": by("cluster", L_new, [list(compare)[0], "Сезонный наивный"])}, axis=1)
print("\nWAPE по кластерам: подвыборка и новые станции")
print(two.round(3).to_string())
two.to_csv(OUT / "newslice_by_cluster.csv")

fig, ax = plt.subplots(figsize=(7, 3.8))
for L, lab, c, ls in [(LB, "подвыборка, выбранная", C_BEST, "-"), (L_new, "новый срез, выбранная", C_2, "-"),
                      (L_new, "новый срез, наивный", C_NAIVE, "--")]:
    col = "Сезонный наивный" if "наивный" in lab else list(compare)[0]
    h = by("hour", L, [col])[col]
    ax.plot(h.index, h.values, color=c, ls=ls, lw=2, marker="o", ms=3, label=lab)
ax.set_yscale("log"); ax.set_xticks(range(0, 24, 2)); ax.set_xlabel("час суток"); ax.set_ylabel("WAPE (лог. шкала)")
log_axis(ax)
ax.set_title("Выбранная модель на новых станциях"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / "05_newslice_by_hour.png"); plt.show()

## 9. Итог

In [ ]:
final = pd.read_csv(OUT / "runs.csv")
print("Прогоны этого ноутбука:")
print(final[["model", "mode", "n_windows", "wape", "wape_ci_lo", "wape_ci_hi", "mase", "cov80", "ms_p95"]].round(4).to_string(index=False))
print(f"\nВыбранная модель: {NAMES[BEST[0]]}, {BEST[1]}")
print(f"  тест ({len(stations)} станций):       WAPE {top['wape']:.4f}  vs наивный {NAIVE:.4f}")
print(f"  новые станции ({len(new_st)}):     WAPE {r_new['wape']:.4f}  vs наивный {n_new['wape']:.4f}")
print(f"  праздники ({len(cols)} станций): WAPE {r_hol['wape']:.4f}  vs наивный {n_hol['wape']:.4f}")
print("\nФайлы:", ", ".join(sorted(p.name for p in OUT.glob("*.csv"))), "| рисунки:", ", ".join(sorted(p.name for p in FIG.glob("*.png"))))